In [12]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI


load_dotenv()


model = ChatGoogleGenerativeAI(model="gemini-3.6-flash")


#State
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

#function definition
def create_outline(state:BlogState)->BlogState:
    #fetch title
    title=state['title']

    #call LLM outline
    prompt=f'Generate a detailed outline for a blog on the topic-{title}'
    outline=model.invoke(prompt).content
    #update state
    state['outline']=outline
    return state

def create_blog(state:BlogState)->BlogState:
    #fetch state
    title=state['title']
    outline=state['outline']

    #call to llm
    prompt=f'Write a detailed blog on the title -{title} using the following outline-{outline}'
    content=model.invoke(prompt).content
    state['content']=content
    return state



#graph
graph=StateGraph(BlogState)

#nodes
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)

#edges
graph.add_edge(START,'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog',END)


#compile
workflow=graph.compile()

#Show Grapgh
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())
#run
initial_state={'title':'Rise of AI in Pakistan'}
final_state=workflow.invoke(initial_state)

print(final_state['outline'])


[{'type': 'text', 'text': 'Here is a comprehensive and detailed blog outline on the **"Rise of AI in Pakistan,"** designed to be informative, engaging, and structured for optimal readability and SEO performance.\n\n---\n\n# **Blog Title Options:**\n1. *Silicon Indus: How AI is Shaping the Future of Pakistan*\n2. *The Rise of AI in Pakistan: Opportunities, Innovations, and Challenges*\n3. *From Bytes to Bots: Unlocking Pakistan’s Artificial Intelligence Potential*\n\n---\n\n## **I. Introduction**\n*   **Catchy Hook:** Open with a striking stat or scenario (e.g., how a Pakistani startup is using AI to save crops, or the global projection of AI’s impact on emerging markets).\n*   **Context:** Brief overview of Pakistan’s rapidly growing tech sector, youthful demographic, and its shift from traditional IT services to deep tech.\n*   **Thesis Statement:** Pakistan stands at the brink of a digital transformation, where Artificial Intelligence is no longer just a buzzword, but a crucial drive